In [ ]:
pip install transformers

In [ ]:
pip install torch

## **BERT**

##### Masked language modeling

se a model trained for masked language modeling, such as:

"bert-base-uncased" or

"distilroberta-base"

In [ ]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model="roberta-base")
sent = "The richest man in India is <mask>."


Device set to use cpu


In [ ]:
results = fill_mask(sent)




In [ ]:
for pred in results:
    print(f"{pred['sequence']} -> score: {pred['score']:.4f}")


The richest man in India is Modi. -> score: 0.2758
The richest man in India is Gandhi. -> score: 0.1588
The richest man in India is Stalin. -> score: 0.0164
The richest man in India is Kejriwal. -> score: 0.0130
The richest man in India is Ram. -> score: 0.0122


## bert using sentimental Analysis

In [ ]:
from transformers import BertTokenizer ,BertForSequenceClassification
from transformers import pipeline

In [ ]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

Device set to use cpu


In [ ]:
sentences=[" I love Ronaldo",
           " I don't like you",
           " The movie big Disaster",
           " Mahesh babu handsome Hunk"]

In [ ]:
for sentence in sentences:
    result = classifier(sentence)
    print(f"Text:  {sentence}")
    print(f"Prediction: {result[0]['label']}, Score: {result[0]['score']:.4f}\n")

Text:   I love Ronaldo
Prediction: POSITIVE, Score: 0.9998

Text:   I don't like you
Prediction: NEGATIVE, Score: 0.9986

Text:   The movie big Disaster
Prediction: NEGATIVE, Score: 0.9997

Text:   Mahesh babu handsome Hunk
Prediction: POSITIVE, Score: 0.9989



# **bert** using Classification

 I am classifing the dataset using bert

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/content/SMSSpamCollection", sep='\t', header=None, names=['label', 'message'])
display(df.head())

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df.shape

(5572, 2)

In [ ]:
x=list(df['message'])
y=list(df['label'])

In [ ]:
y = pd.get_dummies(df['label'], drop_first=True,dtype=int)

In [ ]:
y

,spam
0,0
1,0
2,1
3,0
4,0
...,...
5567,1
5568,0
5569,0
5570,0


In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=9)

# Bert Tokenizer


In [ ]:
from transformers import BertTokenizer
tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
train_encoding=tokenizer(x_train,truncation=True,padding=True)
test_encoding=tokenizer(x_test,truncation=True,padding=True)

converting embedding vectors into objects

It does the following:

Converts train_encoding (which contains input IDs, attention masks, etc.) into a dictionary of tensors.

Pairs these with y_train, which are the labels (targets).

Packs everything into a TensorFlow Dataset object, which is efficient and compatible with model training.



In [ ]:
import tensorflow as tf
train_dataset=tf.data.Dataset.from_tensor_slices((dict(train_encoding),y_train))
test_dataset=tf.data.Dataset.from_tensor_slices((dict(test_encoding),y_test))

In [ ]:
train_dataset

<_TensorSliceDataset element_spec=({'input_ids': TensorSpec(shape=(238,), dtype=tf.int32, name=None), 'token_type_ids': TensorSpec(shape=(238,), dtype=tf.int32, name=None), 'attention_mask': TensorSpec(shape=(238,), dtype=tf.int32, name=None)}, TensorSpec(shape=(1,), dtype=tf.int64, name=None))>

# Loard BERT MODEL for classification

label=2 ,predict spam or ham

In [ ]:
from transformers import BertForSequenceClassification
model=BertForSequenceClassification.from_pretrained('bert-base-uncased',num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


**Training_setup**

**TrainingArguments** is a class used to configure how training should happen when using the Trainer API from Hugging Face.

It's like telling the Trainer: "Here are the rules and strategies to follow during training and evaluation."



In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",                    # Folder to save model checkpoints & final model
    num_train_epochs=3,                        # Train the model for 3 full passes over the training dataset
    per_device_train_batch_size=16,            # Batch size during training per GPU/CPU
    per_device_eval_batch_size=8,             # Batch size during evaluation per GPU/CPU
    eval_strategy="epoch",               # Evaluate the model at the end of each epoch
    save_strategy="epoch",                     # Save the model checkpoint at the end of each epoch
    logging_dir="./logs",                      # Directory to store logs (for TensorBoard or debugging)
    report_to="none"                           # Disable reporting to Weights & Biases
)

| Argument                   | Description                                                                                             |
| -------------------------- | ------------------------------------------------------------------------------------------------------- |
| `model=model`              | Your pre-trained or fine-tuned model (e.g., BERT, RoBERTa, etc.)                                        |
| `args=training_args`       | Training configuration defined using `TrainingArguments` (like batch size, epochs, save strategy, etc.) |
| `train_dataset=train_data` | The training dataset — a `tf.data.Dataset` or PyTorch-style `Dataset`                                   |
| `eval_dataset=test_data`   | The validation (testing) dataset to evaluate after each epoch or step                                   |



Trains your model using the settings in training_args

Automatically handles batching, shuffling, and logging

Saves checkpoints and evaluates model at set intervals

Supports CPU, GPU, TPU

Works with Hugging Face tokenizers and models





In [ ]:
import transformers
print(transformers.__version__)


4.53.2


# evaluate the model

In [ ]:
results=trainer.evaluate()
print(results)

/tmp/ipython-input-16-3001132722.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  item['labels'] = torch.tensor(self.labels.iloc[idx])


{'eval_loss': 0.03391808643937111, 'eval_runtime': 14.2378, 'eval_samples_per_second': 78.313, 'eval_steps_per_second': 9.833, 'epoch': 3.0}


 # Prediction on new Data

In [ ]:
import torch

def predict_sms(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    # Move inputs to the same device as the model
    device = model.device
    inputs = {key: value.to(device) for key, value in inputs.items()}
    outputs = model(**inputs)
    prediction = outputs.logits.argmax(dim=1).item()
    label = "spam" if prediction == 1 else "ham"
    return label

# Example
print(predict_sms("Congratulations! You won a $1000 gift card. Click now!"))

spam
